# 🎯 [DAT10_M02] 오프라인 데이터 입력 유효성 검증 및 로컬 SQLite 트랜잭션 전처리 모듈 구현
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT10/dat10_m02_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT10_M02 |
| **미션 주제** | 오프라인 데이터 입력 유효성 검증 및 로컬 SQLite 트랜잭션 전처리 모듈 구현 |
| **소요 시간** | 약 2.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
전시에는 전술 통신망이 단절된 채 예하 보급소가 입고와 불출을 각자 기록하고, 통신 복구 후 기록을 병합하는 과정에서 중복·누락·충돌·위변조가 빈번히 발생하며, 이를 적시에 통제하지 못하면 재고 왜곡과 방산 공급망 위기 식별이 지연됩니다. 본 프로젝트는 방산 공급망 회복력 연구(미국 방위산업기반 정책연구)의 문제의식을 교육용 가상 자료로 재구성해, 가상 보급소 5곳이 통신 두절 환경에서 오프라인으로 재고를 기록하고 동기화하는 SaaS를 개발하고, 동기화된 데이터에서 공급망 재난 징후(업체 공급 감소, 재고 소진 임박)를 식별하는 End-to-End 군수 소프트웨어 엔지니어링 과제입니다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 오프라인 데이터 입력 유효성 검증 및 로컬 SQLite 트랜잭션 전처리 모듈 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

오프라인 데이터 입력 유효성 검증 및 로컬 트랜잭션 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **형식이 섞인 오프라인 입력 줄을 파싱·검증하고 SQLite 트랜잭션으로 안전하게 적재하는 전처리 모듈을 구현해 처리 속도 1,000건/초 이상, 전처리 오류율 0%를 확인한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **ID\ 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/preprocess/dat10_preprocessor.py 및 전처리 전후 데이터셋` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 전술 환경 특유의 노이즈(스푸핑, 반사파, 센서 단절 등) 제거 타당성

#### 미션 PASS 조건

- 정량 기준: 데이터 정제 처리 속도 1,000건/초 이상, 전처리 오류율 0%


- 진행 기준: 정제 파이프라인 자동화 및 단위 테스트 케이스 5종 통과


- 검증 방법: pytest 기반 전처리 정합성 테스트 및 결측치 보정율 확인


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""서식이 서로 다른 오프라인 입력 문자열 8,000줄(문장형·CSV·JSON·키=값 4종)을 파싱·검증해 SQLite 메모리 DB에 배치 트랜잭션으로 적재하고, 입력 구분(입고·불출·조정)별 동적 임계값으로 이상 수량을 격리하며 중복 재전송 멱등 처리·배치 실패 롤백·처리 속도(건/초)·정답 대조 오류율을 측정한다."""
import json
import re
import sqlite3
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 야전 보급소·업체·품목 자료 생성기 (실제 부대·업체·품목·재고 정보 없음)
# 가상 보급소 5곳이 품목 12종의 입고·불출·조정 트랜잭션을 오프라인으로 기록하고 나중에 동기화한다.
# 공급망 재난 징후(업체 납기 지연, 품목 재고 급감)는 동기화된 자료를 마이닝해 식별한다.
# ---------------------------------------------------------------------------
NODES = ["보급소A", "보급소B", "보급소C", "보급소D", "보급소E"]
SUPPLIERS = ["가상업체1", "가상업체2", "가상업체3", "가상업체4", "가상업체5", "가상업체6"]
CATS = ["일반보급", "정비부품", "의무소모"]
ITEMS = [{"item": f"I{i + 1:02d}", "cat": CATS[i % 3], "supplier": SUPPLIERS[i % 6], "lead_days": 3 + (i * 5) % 12, "use": 4 + (i * 7) % 11, "min_level": 30 + 10 * (i % 5)} for i in range(12)]
ITEM_IDS = [x["item"] for x in ITEMS]
CTXS = ["평시", "야간", "통신불량"]


def make_txns(seed=7, days=7, per_node_day=60, nodes=None):
    """노드별 오프라인 트랜잭션 목록. id는 노드-일련번호(멱등 키), t는 시각(시간 단위), ctx는 입력 상황."""
    rng = np.random.default_rng(seed)
    out = []
    for n in (nodes or NODES):
        seq = 0
        for k in range(days * per_node_day):
            seq += 1
            it = ITEMS[int(rng.integers(len(ITEMS)))]
            t = float(rng.uniform(0, 24 * days))
            ctx = "야간" if (t % 24 >= 20 or t % 24 < 5) else "평시"
            if rng.random() < 0.12:
                ctx = "통신불량"
            r = rng.random()
            if r < 0.22:
                op, q = "in", int(it["use"] * rng.uniform(4, 9))
            elif r < 0.95:
                op, q = "out", int(max(1, rng.normal(it["use"] * 1.5, 3)))
            else:
                op, q = "adjust", int(rng.integers(-5, 6))
            out.append({"id": f"{n}-{seq:05d}", "node": n, "item": it["item"], "op": op, "qty": q, "t": round(t, 2), "ctx": ctx, "supplier": it["supplier"]})
    return out


def stock_of(txns, base=500):
    """품목별 재고 = 기초 재고 + 입고 - 불출 + 조정 (id 중복은 한 번만 반영)."""
    st, seen = {i: base for i in ITEM_IDS}, set()
    for x in txns:
        if x["id"] in seen:
            continue
        seen.add(x["id"])
        st[x["item"]] += x["qty"] if x["op"] in ("in", "adjust") else -x["qty"]
    return st

# ---------------------------------------------------------------------------
# [교육용가상자료] 오프라인 입력 원문 생성: 같은 트랜잭션을 서식 4종 중 하나로 적고, 잡음(이상 수량·형식 깨짐·재전송 중복)을 섞는다.
# ---------------------------------------------------------------------------
rng = np.random.default_rng(21)
base = make_txns(31, 7, per_node_day=230)                      # 5노드 x 7일 x 230건 = 8,050건
OPK = {"in": "입고", "out": "불출", "adjust": "조정"}
lines, truth = [], {}
for i, x in enumerate(base):
    f = i % 4
    if f == 0:
        s = f"[입력] {x['node']} 품목{x['item']} {OPK[x['op']]} {x['qty']}개 ({x['ctx']}) {x['t']}h #{x['id']}"
    elif f == 1:
        s = f"{x['node']},{x['item']},{x['op']},{x['qty']},{x['t']},{x['ctx']},{x['id']}"
    elif f == 2:
        s = json.dumps({k: x[k] for k in ("node", "item", "op", "qty", "t", "ctx", "id")}, ensure_ascii=False)
    else:
        s = f"품목 {x['item']} / {OPK[x['op']]} / {x['qty']}개 / {x['node']} / t={x['t']} / ctx={x['ctx']} / id={x['id']}"
    lines.append(s); truth[x["id"]] = x
noise_ids = set()
for i in rng.choice(len(lines), 120, replace=False):          # 잡음 주입: 이상 수량 40, 깨진 줄 40, 존재하지 않는 품목 40
    x = base[i]
    if i % 3 == 0:
        lines[i] = lines[i].replace(str(x["qty"]), "9999", 1) if x["op"] != "adjust" else lines[i]; noise_ids.add(x["id"]) if "9999" in lines[i] else None
    elif i % 3 == 1:
        lines[i] = lines[i][: len(lines[i]) // 2]; noise_ids.add(x["id"])
    else:
        lines[i] = lines[i].replace(x["item"], "I99"); noise_ids.add(x["id"])
dups = [lines[i] for i in rng.choice(len(lines), 200, replace=False)]   # 통신 복구 후 재전송된 중복 줄 200개
lines = lines + dups

# ---------------------------------------------------------------------------
# 파서: 서식별 정규식 -> 표준 레코드 (해석 실패는 예외)
# ---------------------------------------------------------------------------
IV = {v: k for k, v in OPK.items()}
P1 = re.compile(r"\[입력\]\s*(보급소[A-E])\s*품목(I\d{2})\s*(입고|불출|조정)\s*(-?\d+)개\s*\((\S+)\)\s*([\d.]+)h\s*#(\S+)")
P4 = re.compile(r"품목\s*(I\d{2})\s*/\s*(입고|불출|조정)\s*/\s*(-?\d+)개\s*/\s*(보급소[A-E])\s*/\s*t=([\d.]+)\s*/\s*ctx=(\S+)\s*/\s*id=(\S+)")


def parse(s):
    s = s.strip()
    if s.startswith("["):
        m = P1.match(s)
        if m:
            return {"node": m[1], "item": m[2], "op": IV[m[3]], "qty": int(m[4]), "ctx": m[5], "t": float(m[6]), "id": m[7]}
    elif s.startswith("{"):
        return json.loads(s)
    elif s.startswith("품목"):
        m = P4.match(s)
        if m:
            return {"item": m[1], "op": IV[m[2]], "qty": int(m[3]), "node": m[4], "t": float(m[5]), "ctx": m[6], "id": m[7]}
    else:
        p = s.split(",")
        if len(p) == 7:
            return {"node": p[0], "item": p[1], "op": p[2], "qty": int(p[3]), "t": float(p[4]), "ctx": p[5], "id": p[6]}
    raise ValueError("서식 해석 불가")


def valid(r):
    if r["item"] not in ITEM_IDS or r["node"] not in NODES or r["op"] not in OPK or r["ctx"] not in CTXS:
        return "허용값 오류"
    if r["op"] != "adjust" and r["qty"] <= 0:
        return "수량 오류"
    if not re.match(r"^보급소[A-E]-\d{5}$", r["id"]) or not r["id"].startswith(r["node"]):
        return "ID 오류"
    return None


# 1) 파싱 + 검증 + 동적 임계값
t0 = time.perf_counter()
parsed, rejected = [], []
for s in lines:
    try:
        r = parse(s); e = valid(r)
        (rejected if e else parsed).append((r, e) if e else r)
    except (ValueError, KeyError, json.JSONDecodeError, IndexError) as ex:
        rejected.append((s, "해석 실패"))
df = pd.DataFrame(parsed)
before = len(df)
df = df.drop_duplicates("id")
th = {}
for op, g in df[df["op"] != "adjust"].groupby("op"):
    med = g["qty"].median(); mad = (g["qty"] - med).abs().median() * 1.4826; th[op] = (max(med - 6 * mad, 1), med + 6 * mad)
fixed = (df["op"] != "adjust") & (df["qty"] > 100)
dyn = pd.Series([(op != "adjust") and not (th[op][0] <= q <= th[op][1]) for op, q in zip(df["op"], df["qty"])], index=df.index)
susp = dyn.to_numpy()
clean = df[~dyn].copy()
dt_parse = time.perf_counter() - t0
rej_cnt = pd.Series([e for _, e in rejected]).value_counts().to_dict()
print(f"[파싱·검증] 입력 {len(lines)}줄(재전송 중복 200 포함): 해석 실패·허용값 오류 {len(rejected)}줄 {rej_cnt}, 통과 {before}줄, 중복 ID {before - len(df)}줄 제거")
print("구분별 동적 임계값(수량):", ", ".join(f"{OPK[k]} {lo:.0f}~{hi:.0f}" for k, (lo, hi) in th.items()))
true_bad = df["id"].isin(noise_ids).to_numpy()
for name, fl in [("고정 임계값(100개 초과)", fixed.to_numpy()), ("구분별 동적 임계값", susp)]:
    print(f"  {name:20s}: 이상 판정 {int(fl.sum())}건, 이상 수량 {int((fl & true_bad).sum())}/{int(true_bad.sum())} 검출, 정상 입고인데 이상 판정(오탐) {int((fl & ~true_bad).sum())}건")

# 2) SQLite 배치 트랜잭션 적재(멱등: 같은 ID는 한 번만), 배치 실패 시 롤백 후 행 단위로 격리
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE txn (id TEXT PRIMARY KEY, node TEXT NOT NULL, item TEXT NOT NULL, op TEXT NOT NULL CHECK (op IN ('in','out','adjust')), qty INTEGER NOT NULL, t REAL NOT NULL CHECK (t >= 0), ctx TEXT NOT NULL)")
db.execute("CREATE TABLE quarantine (id TEXT, reason TEXT)")
rows = [(r["id"], r["node"], r["item"], r["op"], int(r["qty"]), float(r["t"]), r["ctx"]) for r in clean.to_dict("records")]
rows[777] = (rows[777][0], rows[777][1], rows[777][2], "sell", 5, 1.0, "평시")      # 배치 안에 제약 위반 1건 주입
t1 = time.perf_counter(); inserted = rolled = 0
for b in range(0, len(rows), 500):
    batch = rows[b:b + 500]
    try:
        with db:
            db.executemany("INSERT INTO txn VALUES (?,?,?,?,?,?,?) ON CONFLICT(id) DO NOTHING", batch)
        inserted += len(batch)
    except sqlite3.IntegrityError:
        rolled += 1                                            # 배치 전체 롤백 후 행 단위 재시도
        for r in batch:
            try:
                with db:
                    db.execute("INSERT INTO txn VALUES (?,?,?,?,?,?,?) ON CONFLICT(id) DO NOTHING", r)
                inserted += 1
            except sqlite3.IntegrityError as e:
                with db:
                    db.execute("INSERT INTO quarantine VALUES (?,?)", (r[0], "제약 위반"))
dt_db = time.perf_counter() - t1
n_db = db.execute("SELECT COUNT(*) FROM txn").fetchone()[0]
print(f"\n[SQLite 적재] 500건 배치 {len(rows) // 500 + 1}개: 롤백 {rolled}회(제약 위반 1건 -> 행 단위 격리), 적재 {n_db}건, 격리 테이블 {db.execute('SELECT COUNT(*) FROM quarantine').fetchone()[0]}건")
db.execute("INSERT INTO txn VALUES (?,?,?,?,?,?,?) ON CONFLICT(id) DO NOTHING", rows[0]); n2 = db.execute("SELECT COUNT(*) FROM txn").fetchone()[0]
print(f"같은 ID를 다시 넣어도 행 수 {n2}건 유지(멱등)")

# 3) 정답 대조: 적재된 행의 필드가 원본 트랜잭션과 같은가, 속도
errs = 0; chk = 0
for (i, node, item, op, qty, t, ctx) in db.execute("SELECT * FROM txn"):
    x = truth[i]; chk += 1
    errs += (x["node"], x["item"], x["op"], x["qty"], x["ctx"]) != (node, item, op, qty, ctx) or abs(x["t"] - t) > 1e-6
total = dt_parse + dt_db
print(f"\n[전처리 정확도] 적재 {chk}건 중 원본과 불일치 {errs}건 -> 전처리 오류율 {errs / chk:.2%} (목표 0%: {'달성' if errs == 0 else '미달'})")
print(f"처리 속도 {len(lines) / total:,.0f}건/초 (파싱·검증 {dt_parse:.2f}초 + DB 적재 {dt_db:.2f}초, 목표 1,000건/초 이상: {'달성' if len(lines) / total >= 1000 else '미달'})")
print(f"원본 {len(base)}건 중 잡음 주입 {len(noise_ids)}건과 격리 {int(susp.sum())}건을 제외하고 적재 {n_db}건 -> 격리된 건은 담당자가 원본 확인 후 재입력")
print("※ 이상 판정된 줄은 삭제하지 않고 격리하며, 임계값은 입력 구분별 중앙값±6MAD로 자료에서 조정한다. 실제 서식에 맞는 정규식은 현장에서 보완한다.")

plt.figure(figsize=(8, 4))
for op, g in df[df["op"] != "adjust"].groupby("op"):
    plt.hist(g["qty"].clip(upper=200), bins=40, alpha=.5, label=op)
plt.axvline(100, color="k", ls="--"); plt.legend(); plt.xlabel("qty"); plt.title("Quantity by operation (dashed: fixed threshold 100)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
[파싱·검증] 입력 8250줄(재전송 중복 200 포함): 해석 실패·허용값 오류 84줄 {'허용값 오류': 44, '해석 실패': 40}, 통과 8166줄, 중복 ID 198줄 제거
구분별 동적 임계값(수량): 입고 1~221, 불출 1~56
  고정 임계값(100개 초과)     : 이상 판정 139건, 이상 수량 34/34 검출, 정상 입고인데 이상 판정(오탐) 105건
  구분별 동적 임계값          : 이상 판정 34건, 이상 수량 34/34 검출, 정상 입고인데 이상 판정(오탐) 0건

[SQLite 적재] 500건 배치 16개: 롤백 1회(제약 위반 1건 -> 행 단위 격리), 적재 7933건, 격리 테이블 1건
같은 ID를 다시 넣어도 행 수 7933건 유지(멱등)

[전처리 정확도] 적재 7933건 중 원본과 불일치 0건 -> 전처리 오류율 0.00% (목표 0%: 달성)
처리 속도 48,683건/초 (파싱·검증 0.13초 + DB 적재 0.04초, 목표 1,000건/초 이상: 달성)
원본 8050건 중 잡음 주입 116건과 격리 34건을 제외하고 적재 7933건 -> 격리된 건은 담당자가 원본 확인 후 재입력
※ 이상 판정된 줄은 삭제하지 않고 격리하며, 임계값은 입력 구분별 중앙값±6MAD로 자료에서 조정한다. 실제 서식에 맞는 정규식은 현장에서 보완한다.
```
</details>
